In [5]:
import sys, os, glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import ascii, fits
from astropy import table
import csv
from reproject import reproject_interp
from scipy.interpolate import RegularGridInterpolator
from astropy.convolution import convolve_fft
from astropy.wcs import WCS

In [ ]:
import ifu_convolution as ifuc
# Example workflow to convolve NIRSpec G140M cube to the longset wavelength slice in MIRI ch4 short

In [2]:
# Input NIRSpec science cube
cube_file = (
    '/data/jwst/GO3435/NIRSpec/obs4/stage3/f100lp_g140m-f100lp_s3d.fits'
)

# Test output directory
test_dir = "/home/hkoziol/Desktop/m51/ifu_test"
os.makedirs(test_dir, exist_ok=True)

In [3]:
# Outputs
source_psf_file = os.path.join(test_dir, "g140m_psf.fits")
kernel_dir = os.path.join(test_dir, "kernels")
kernel_cube_file = os.path.join(test_dir, "kernel_cube.fits")
convolved_cube_file = os.path.join(test_dir, "g140m_convolved.fits")

In [6]:
from astropy import units as u
# Generate the PSF at every wavelength slice of the NIRSpec cube 
with fits.open(cube_file) as cube:
    sci = cube["SCI"]
    wavelengths = ifuc.get_cube_wavelengths(cube, unit=u.m)

    pixel_scale = ifuc.get_pixscale(sci)

    print("Cube shape:", sci.data.shape)
    print("Pixel scale:", pixel_scale, "arcsec/pixel")
    print("Wavelength range:", wavelengths.min(), "to", wavelengths.max())
    
    ifuc.generate_psf(
        instrument="NIRSpec",
        resolution="G140M",
        output_file=source_psf_file,
        cube_file=cube_file,
        filter_name="F100LP",
        disperser="G140M",
        mode="IFU",
        fov_arcsec=9.0,
        oversample=5,
        pixel_scale=pixel_scale,
        overwrite=True,
    )

Set DATE-AVG to '2024-04-02T19:28:11.528' from MJD-AVG.
Set DATE-END to '2024-04-02T20:07:17.736' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     3.594497 from OBSGEO-[XYZ].
Set OBSGEO-H to 1246154887.137 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


Cube shape: (1447, 241, 85)
Pixel scale: 0.04999999999999968 arcsec/pixel
Wavelength range: 9.703180286160204e-07 m to 1.8899740453634867e-06 m


Wrote IFU PSF cube: /home/hkoziol/Desktop/m51/ifu_test/g140m_psf.fits


In [7]:
# Load the generated source PSF cube
with fits.open(source_psf_file) as hdul:
    hdul.info()

    # Check which extension contains the PSF data.
    source_psf_cube = np.asarray(hdul['DET_SAMP'].data)

print("Source PSF data shape:", source_psf_cube.shape)

Filename: /home/hkoziol/Desktop/m51/ifu_test/g140m_psf.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  OVERSAMP      1 PrimaryHDU   34816   (500, 500, 1447)   float64   
  1  DET_SAMP      1 ImageHDU     34818   (100, 100, 1447)   float64   
  2  OVERDIST      1 ImageHDU     36268   (500, 500, 1447)   float64   
  3  DET_DIST      1 ImageHDU     36269   (100, 100, 1447)   float64   
Source PSF data shape: (1447, 100, 100)


In [8]:
with fits.open(target_psf_file) as hdul:
    hdul.info()

    # Open the last slice in the target PSF cube
    target_psf = np.asarray(hdul['DET_SAMP'].data[-1], dtype=float)

print("Target PSF shape:", target_psf.shape)

Filename: /home/hkoziol/Desktop/m51/PSF_cubes/miri_ch4_short.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  OVERSAMP      1 PrimaryHDU    9846   (170, 170, 542)   float64   
  1  DET_SAMP      1 ImageHDU      9848   (34, 34, 542)   float64   
  2  OVERDIST      1 ImageHDU     10937   (170, 170, 542)   float64   
  3  DET_DIST      1 ImageHDU     10938   (34, 34, 542)   float64   
Target PSF shape: (34, 34)


In [9]:
source_pixscale = 0.05  # EXAMPLE: source PSF pixel scale, arcsec/pixel
target_pixscale = (9.72222205665376E-05*3600)  # EXAMPLE: target PSF pixel scale, arcsec/pixel

target_fwhm = ifuc.fit_2d_gaussian(
    target_psf,
)

print("Target FWHM:", target_fwhm, "arcsec")

ifuc.make_kernel_cube(
    source_cube=source_psf_cube,
    source_wavelengths=wavelengths,
    target_psf=target_psf,
    target_fwhm=target_fwhm,
    source_name="NIRSpec_G140M",
    output_dir=kernel_dir,
    source_pixscale=source_pixscale,
    target_pixscale=target_pixscale,
    target_name="target",
)

Target FWHM: 1.993597186246464 arcsec
Processing source and target PSFs
Skipping resample (pixel scales match)
Centroiding
Circularizing
Resizing
  final shape: (237, 237)
Resampling
Centroiding
Circularizing
Resizing
  final shape: (237, 237)
k_h too large. Something is probably up with kernel generation...
Finished kernel 0: wavelength=0.9703180286160203
Processing source and target PSFs
Skipping resample (pixel scales match)
Centroiding
Circularizing
Resizing
  final shape: (237, 237)
Resampling
Centroiding
Circularizing
Resizing
  final shape: (237, 237)
k_h too large. Something is probably up with kernel generation...
Finished kernel 1: wavelength=0.9709540286276023
Processing source and target PSFs
Skipping resample (pixel scales match)
Centroiding
Circularizing
Resizing
  final shape: (237, 237)
Resampling
Centroiding
Circularizing
Resizing
  final shape: (237, 237)
k_h too large. Something is probably up with kernel generation...
Finished kernel 2: wavelength=0.9715900286391842

'/home/hkoziol/Desktop/m51/ifu_test/kernels'

In [10]:
ifuc.stack_kernel_cube(
    kernel_dir=kernel_dir,
    output_file=kernel_cube_file,
    pixscale=source_pixscale,
    wavelengths=wavelengths,
    overwrite=True,
)


Padding all kernels to 231 x 231
Wrote kernel cube: /home/hkoziol/Desktop/m51/ifu_test/kernel_cube.fits


array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.

In [11]:
target_fwhm = ifuc.fit_2d_gaussian(
    target_psf,
    pixscale=target_pixscale,
)

print("Target FWHM:", target_fwhm, "arcsec")
final_res_path = '/home/hkoziol/Desktop/m51/v2p0/NE_sky_ch4-short_s3d.fits'
final_res_file = fits.open(final_res_path)
wcs_2d = WCS(final_res_file['SCI'].header).celestial
ny, nx = final_res_file['SCI'].shape[-2:]
grid_ch4_short = (wcs_2d, (ny, nx))
ifuc.do_jwst_convolution_3d_astropy(
    file_in=cube_file,
    file_out=convolved_cube_file,
    file_kernel=kernel_cube_file,
    output_grid = grid_ch4_short
)

Target FWHM: 0.6977590033034969 arcsec


Set DATE-AVG to '2024-05-31T00:02:57.457' from MJD-AVG.
Set DATE-END to '2024-05-31T00:35:51.619' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -32.909368 from OBSGEO-[XYZ].
Set OBSGEO-H to 1706867093.776 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-04-02T19:28:11.528' from MJD-AVG.
Set DATE-END to '2024-04-02T20:07:17.736' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     3.594497 from OBSGEO-[XYZ].
Set OBSGEO-H to 1246154887.137 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2024-04-02T19:28:11.528' from MJD-AVG.
Set DATE-END to '2024-04-02T20:07:17.736' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to     3.594497 from OBSGEO-[XYZ].
Set OBSGEO-H to 1246154887.137 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
